In [1]:
import torch
import numpy as np
import os
import soundfile as sf 
import librosa
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [2]:
# Function that finds all files with a certain extension in a particular folder
def find_all_files(folders, ext, check_sub_folders=1):
    """Find all files with a .txt extension in all folders in folder list (and subfolders if check_sub_folders = 1 (default))"""
    all_files = []
    if isinstance(folders, str):
        # Make sure input is a list of folders
        folders = [folders]
    for f, folder in enumerate(folders):
        for subfolder in os.listdir(folder):

            if os.path.isdir(os.path.join(folder, subfolder)):
                if check_sub_folders:
                    folders.append(os.path.join(folder, subfolder))                
            elif subfolder.endswith(ext):
                all_files.append(os.path.join(folder,subfolder))
    return all_files

In [3]:
# Function that pads or cuts a spectrogram to a certain size, adapted from avgn
def pad_spectrogram(spectrogram, pad_length):
    """ Pads a spectrogram to being a certain length
    """
    excess_needed = pad_length - np.shape(spectrogram)[1]
    pad_left = np.floor(float(excess_needed) / 2).astype("int")
    pad_right = np.ceil(float(excess_needed) / 2).astype("int")
    if excess_needed >= 0:
        return np.pad(
            spectrogram, [(0, 0), (pad_left, pad_right)], "constant", constant_values=0
        )
    else:
        pad_left = np.floor(float(excess_needed) / 2).astype("int")
        pad_right = np.ceil(float(excess_needed) / 2).astype("int")
        return spectrogram[:,-1*pad_left:pad_right]

## Setup

In [4]:
# Constant variables
fmin = 1000
fmax = 10000 # Max frequency cutoff
spec_shape = [128, 128] #Shape of spectrogram
n_fft = 2048
hop_length = int(n_fft/2)


In [5]:
# Find all the audio files
data_folder = r"C:\Users\grijseels\Documents\Github\AnimalBehaviourLiveWorkshop\test_data"
data_files = find_all_files(data_folder, '.flac')